In [ ]:
import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import GRB

In [ ]:
###### Some functions

def coordinate(i,j,GridNumber):
    x = 1/(GridNumber)*(i-1)
    y = 1/(GridNumber)*(j-1)
    return x, y

def coordinate_10(i,j):
    x = 0.1*(i-1)
    y = 0.1*(j-1)
    return x, y

def coordinate_20(i,j):
    x = 0.05*(i-1)
    y = 0.05*(j-1)
    return x, y

def coordinate_step(i,j,GridNumber):
    x = i
    y = j
    return x, y

def count_duplicates(row):
    value_counts = row.value_counts()
    return [value for value in value_counts.index]

def build_index_sets(row):
    index_sets = {}
    for index, value in enumerate(row):
        if value not in index_sets:
            index_sets[value] = []
        index_sets[value].append(index)
    return index_sets

In [ ]:
###### Construct the matrix

demand_grid_number = 10
hub_grid_number = 20
Totalofpair = (pow(demand_grid_number+1,2)+1)*pow(demand_grid_number+1,2)/2
print("The total number of pair is:", Totalofpair)


ColumnsName=[f'P{i,j}' for i in range(1, hub_grid_number + 2) for j in range(1, hub_grid_number + 2)]
# print(ColumnsName)

Rowindex = []

Dia_index = []
Dia_counter = 0

for i in range(1, demand_grid_number + 2):
    for j in range(1, demand_grid_number + 2):
        for l in range(j,demand_grid_number+2):
            if l == j:
                Dia_index.append(Dia_counter)

            temp = f'O{i,j}_D{i,l}'
            Rowindex.append(temp)
            Dia_counter = Dia_counter+1

        for k in range(i+1,demand_grid_number + 2):
            for l in range(1,demand_grid_number + 2):
                temp = f'O{i,j}_D{k,l}'
                Rowindex.append(temp)
                Dia_counter = Dia_counter+1


df = pd.DataFrame( index= Rowindex, columns=ColumnsName)




for i in range(1, demand_grid_number + 2):
    for j in range(1, demand_grid_number + 2):
        for l in range(j,demand_grid_number+2):

            
        
            for m in range(1, hub_grid_number + 2):
                for n in range(1, hub_grid_number + 2):

                    Oi,Oj = coordinate(i,j,demand_grid_number)
                    Di,Dj = coordinate(i,l,demand_grid_number)
                    Pi,Pj = coordinate(m,n,hub_grid_number)
                    
                
                    #df.loc[f'O{i,j}_D{i,l}',f"P{m,n}"] = np.abs(Oi-Pi)+np.abs(Oj-Pj)+np.abs(Di-Pi)+np.abs(Dj-Pj)

                    df.loc[f'O{i,j}_D{i,l}',f"P{m,n}"] = round((np.abs(Oi-Pi)+np.abs(Oj-Pj)+np.abs(Di-Pi)+np.abs(Dj-Pj))*100)
                    

        for k in range(i+1,demand_grid_number + 2):
            for l in range(1,demand_grid_number + 2):

                for m in range(1, hub_grid_number + 2):
                    for n in range(1, hub_grid_number + 2):

                        Oi,Oj = coordinate(i,j,demand_grid_number)
                        Di,Dj = coordinate(k,l,demand_grid_number)
                        Pi,Pj = coordinate(m,n,hub_grid_number)

                        #df.loc[f'O{i,j}_D{k,l}',f"P{m,n}"] = np.abs(Oi-Pi)+np.abs(Oj-Pj)+np.abs(Di-Pi)+np.abs(Dj-Pj)
                        df.loc[f'O{i,j}_D{k,l}',f"P{m,n}"] = round((np.abs(Oi-Pi)+np.abs(Oj-Pj)+np.abs(Di-Pi)+np.abs(Dj-Pj))*100)
                        
                        
    
    ## Print reduced Arc
                        
print("the Dia_index is",Dia_index)



# Apply the function to each row
print(df.head())
result = df.apply(count_duplicates, axis=1)

#print(result)

print(result.head())

NUnique = 0

for i in range(len(result)):
    NUnique = NUnique + len(result.iloc[i])

print("total number of unique value is", NUnique)
TotalArc = Totalofpair* len(ColumnsName)
print("Total number of arc is:", TotalArc)

print("the percentage of Arc after reduction", NUnique/TotalArc)


In [ ]:
###### generate some parameters

rows_index_sets = df.apply(build_index_sets, axis=1)

## Build Kmax

Kmax = []
for i in range(len(rows_index_sets)):
    Kmax.append(len(rows_index_sets[i]))

#print("Kmax",Kmax)

## Calculate distance d_ik

def distance_calculation (data_frame):
    distance={}
    for i in range (0, len(data_frame)):
        k_value = list(sorted(data_frame[i].keys()))
        for k in range (len(data_frame[i])):
            distance[(i,k)] = k_value[k] 
    return distance

distance = distance_calculation(rows_index_sets)


## Get S_ik

def Set_obtain (data_frame):
    Set_ties={}
    for i in range (0, len(data_frame)):
        sorted_dict = dict(sorted(data_frame[i].items()))
        index = list(sorted(data_frame[i].keys()))
        for k in range (len(data_frame[i])):            
            Set_ties[(i,k)] = sorted_dict[index[k]]
    return Set_ties

Set_ties = Set_obtain(rows_index_sets)


In [ ]:


###### Build Gurobi 


m = gp.Model()
Pnumber = 2
m.Params.MIPGap = 0

# variables: t_ij, y_j, 

Pair_range = range(len(rows_index_sets))
Location_range = range(len(ColumnsName))

t = {}
for i in Pair_range:
    for k in range(Kmax[i]):
        t[i,k] = m.addVar(vtype=GRB.CONTINUOUS,name = f"t_+{i,k}")

y = m.addVars(Location_range ,vtype=GRB.BINARY, name="y_i")


# Obejective function




Non_Dia_index = [item for item in Pair_range if item not in Dia_index]

obj = gp.quicksum(

        distance[i,0] + gp.quicksum( (distance[i,k+1]-distance[i,k])*t[i,k] for k in range(Kmax[i]-1))
        for i in Pair_range
    ) + gp.quicksum(

        distance[i,0] + gp.quicksum( (distance[i,k+1]-distance[i,k])*t[i,k] for k in range(Kmax[i]-1))
        for i in Non_Dia_index
    ) 
        

# Constraints

m.addConstr(gp.quicksum(y[j] for j in Location_range) == Pnumber,name = "plocation")

for i in Pair_range:
    m.addConstr(t[i,0]+ gp.quicksum(y[j] for j in Set_ties[i,0]) >= 1,name = f"constraint1_{i}")

for i in Pair_range:
    for k in range(1,Kmax[i]):
        m.addConstr(t[i,k]+ gp.quicksum(y[j] for j in Set_ties[i,k]) >= t[i,k-1],name = f"constraint2_{i}")



print("start the optimization")

m.setObjective(obj, GRB.MINIMIZE)
m.optimize()
runtime = m.Runtime
print("running time is ", runtime)

hubs = []
# yvalues = m.getAttr('x',y)
for k,y in y.items():
    if y.x > 0:
        hubs.append(k)

for i in hubs:
     print(ColumnsName[i])

# print(yvalues)

